# Import libraries

In [1]:
from pyspark.sql import SparkSession
import pyspark
import os

# Variables

In [2]:
HADOOP_VER = "3.4.1"
ROOT_PATH = "s3a://pd2/cityenjoyer"

# Initialize PySpark
***Must be connected to UCM's network or VPN***

In [3]:
if 'spark' in locals():
    spark.stop()

spark = SparkSession.builder \
    .appName("MinioFinalAttempt") \
    .config("spark.jars.packages", f"org.apache.hadoop:hadoop-aws:{HADOOP_VER}") \
    .config("spark.hadoop.fs.s3a.endpoint", "https://minio.fdi.ucm.es/") \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY")) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.vectored.active", "false") \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "100") \
    .getOrCreate()

spark.sparkContext._jsc.hadoopConfiguration()

:: loading settings :: url = jar:file:/home/QuiSioU/UCM/y3/PD2/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/QuiSioU/.ivy2.5.2/cache
The jars for the packages stored in: /home/QuiSioU/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dfa4ea73-06f7-4b39-9e74-28ec78a5a2cb;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
:: resolution report :: resolve 152ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.1.3.Final from central in [default]
	software.amazon.awssdk#bundle;2.24.6 from central in [default]
	---------------------------------------------------------------------
	|     

JavaObject id=o57

# Analysis

In [13]:
green_df = spark.read.parquet(f"{ROOT_PATH}/green_taxi_21-25.parquet")
green_df.printSchema()

root
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- store_and_fwd_flag: boolean (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- trip_type: string (nullable = true)
 |-- VendorID: string (nullable = true)
 |-- ehail_fee: double (nullable = true)



In [25]:
filter_neg_value = \
    (green_df.passenger_count < 0) | \
    (green_df.trip_distance < 0) | \
    (green_df.tip_amount < 0) | \
    (green_df.total_amount < 0)

In [28]:
n_rows = green_df.count()
n_neg_rows = green_df.filter(filter_neg_value).count()

In [31]:
print(f"Procentage of 'negative-value' outliers: {n_neg_rows / n_rows * 100:.2f}%")

Procentage of 'negative-value' outliers: 0.27%
